# 모기 비행 궤적 예측 — Private 2위 (Private 0.7031 / Public 0.7022)

데이콘 모기 비행 궤적 예측 대회 · 2차 평가 제출 코드.

- 지표: R-Hit@1cm (1cm 이내 hit 비율). 입력 11개 시점 3D → +80ms 예측
- 실행: 위에서부터 Run All → §7이 최종 제출 `submission_v157_ens3a0.4.csv` (Private 0.7031) 생성. **단일 .ipynb로 완결 — 외부 데이터·산출물 불필요**
- 입출력 상대경로 · UTF-8
- 환경: Windows 11 / Python 3.11.2 / numpy 2.2.6 · pandas 2.2.3 · scipy 1.15.3 · scikit-learn 1.6.1 · torch 2.7.1 · tqdm 4.67.1 (requirements.txt)
- 구성: §1 공통 코어 · §2~§5 앙상블 멤버(대표 구현) · §6 블렌드 레시피 · §7 최종 제출 복원
- 상세 방법론: 첨부 PDF


## §1. 환경 · 데이터 · 공통 코어 (Kalman 잔차 / canonical frame / scalar features)


In [1]:
from __future__ import annotations
import gc, glob, os, random, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm


def seed_everything(seed=0):
    """재현성을 위한 시드 고정 (python/numpy/torch)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def find_root(start: Path) -> Path:
    """data/ 폴더가 있는 프로젝트 루트를 cwd에서 위로 탐색 (상대경로 기반)."""
    p = start.resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists():
            return q
    return p


seed_everything(0)
ROOT = find_root(Path.cwd())
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "data" / "cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[setup] device = {device} | ROOT = {ROOT.name}")

# 개발/검증 환경 출력 (대회 규칙: 라이브러리 로딩·버전 기재)
import sys as _sys, platform as _plat, scipy as _scipy, sklearn as _sk
print(f"[env] {_plat.system()} {_plat.release()} | python {_sys.version.split()[0]} | "
      f"numpy {np.__version__} pandas {pd.__version__} scipy {_scipy.__version__} "
      f"sklearn {_sk.__version__} torch {torch.__version__}")

[setup] device = cpu | ROOT = 모기 비행 궤적 예측 AI 경진대회
[env] Windows 10 | python 3.11.2 | numpy 2.2.6 pandas 2.2.3 scipy 1.15.3 sklearn 1.6.1 torch 2.7.1+cpu


In [2]:
DT = 0.040
T_PRED = 0.080
T_OBS = np.arange(-400, 1, 40) / 1000.0


def load_data() -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """train/test 궤적(각 10000개, 11시점×3D)과 라벨·sample_submission 로드."""
    cache = CACHE_DIR / "xtrain_xtest.npz"
    train_files = sorted(glob.glob(str(DATA_DIR / "train" / "*.csv")))
    test_files  = sorted(glob.glob(str(DATA_DIR / "test" / "*.csv")))
    labels = pd.read_csv(DATA_DIR / "train_labels.csv")
    sub    = pd.read_csv(DATA_DIR / "sample_submission.csv")
    train_ids = [os.path.splitext(os.path.basename(f))[0] for f in train_files]
    y_train = labels.set_index("id").loc[train_ids][["x", "y", "z"]].values.astype(np.float64)

    if cache.exists():
        nc = np.load(cache)
        X_train, X_test = nc["X_train"], nc["X_test"]
        print(f"[data] cache 로드: X_train {X_train.shape}")
    else:
        def _read(f): return pd.read_csv(f)[["x", "y", "z"]].values
        def _load(files, desc, workers=8):
            arrays = [None] * len(files)
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futs = {ex.submit(_read, f): i for i, f in enumerate(files)}
                for fu in tqdm(futs, desc=desc, total=len(futs)):
                    arrays[futs[fu]] = fu.result()
            return np.stack(arrays, axis=0).astype(np.float64)
        X_train = _load(train_files, "train")
        X_test  = _load(test_files, "test")
        np.savez(cache, X_train=X_train, X_test=X_test)
        print(f"[data] cache 저장: {cache}")

    assert X_train.shape[0] == y_train.shape[0] == 10000
    return X_train, X_test, y_train, sub

In [3]:
def kalman_predict(X, sigma_obs=0.30e-3, sigma_proc=1.0, P0=1.0):
    """등속 칼만필터로 +80ms 위치를 외삽 (여러 멤버가 공유하는 잔차 baseline)."""
    N, T, _ = X.shape
    F = np.array([[1, DT], [0, 1]])
    F_pred = np.array([[1, T_PRED], [0, 1]])
    Q = sigma_proc ** 2 * np.array([[DT**4/4, DT**3/2], [DT**3/2, DT**2]])
    R = sigma_obs ** 2
    pred = np.zeros((N, 3))
    for j in range(3):
        z_all = X[:, :, j]
        state = np.zeros((N, 2)); state[:, 0] = z_all[:, 0]
        P = np.eye(2) * P0
        for t in range(1, T):
            state = state @ F.T
            P = F @ P @ F.T + Q
            innov = z_all[:, t] - state[:, 0]
            S = P[0, 0] + R; K = P[:, 0] / S
            state = state + np.outer(innov, K)
            P = P - np.outer(K, P[0, :])
        pred[:, j] = (state @ F_pred.T)[:, 0]
    return pred


def get_kalman(X_train, X_test):
    """Kalman 예측을 계산하고 캐시."""
    cache = CACHE_DIR / "kalman.npz"
    if cache.exists():
        kc = np.load(cache)
        print("[kalman] cache 로드")
        return kc["kalman_train"], kc["kalman_test"], kc["kalman_train_alt"]
    print("[kalman] 계산 중 (10~30초)…")
    k_tr = kalman_predict(X_train, sigma_obs=0.30e-3, sigma_proc=1.0)
    k_te = kalman_predict(X_test,  sigma_obs=0.30e-3, sigma_proc=1.0)
    k_tr_alt = kalman_predict(X_train, sigma_obs=1.0e-3, sigma_proc=1.0)
    np.savez(cache, kalman_train=k_tr, kalman_test=k_te, kalman_train_alt=k_tr_alt)
    print(f"[kalman] cache 저장: {cache}")
    return k_tr, k_te, k_tr_alt

In [4]:
def noise_poly2(X):
    """2차 다항 피팅 잔차로 관측 노이즈를 추정."""
    V = np.vander(T_OBS, 3, increasing=False)
    out = np.zeros(X.shape[0])
    for j in range(3):
        coef = np.linalg.lstsq(V, X[:, :, j].T, rcond=None)[0]
        out += (X[:, :, j] - (V @ coef).T).std(axis=1)
    return out / 3


def noise_savgol(X, w=5, p=2):
    """Savitzky-Golay 평활 잔차로 관측 노이즈를 추정."""
    Xs = savgol_filter(X, window_length=w, polyorder=p, axis=1)
    return (X - Xs).std(axis=1).mean(axis=-1)


def noise_loo_subset(X, sample_idx):
    """Leave-one-out cubic-spline 잔차로 노이즈를 추정 (subset만)."""
    sav = noise_savgol(X)
    out = sav.copy()
    idx_all = np.arange(len(T_OBS))
    for i in tqdm(sample_idx, desc="LOO spline subset"):
        s = 0.0
        for k in range(1, len(T_OBS) - 1):
            mask = idx_all != k
            for j in range(3):
                cs = CubicSpline(T_OBS[mask], X[i, mask, j])
                s += (X[i, k, j] - cs(T_OBS[k])) ** 2
        out[i] = np.sqrt(s / ((len(T_OBS) - 2) * 3))
    return out


def cos_safe(a, b):
    """수치적으로 안전한 코사인 유사도."""
    na = np.linalg.norm(a, axis=-1); nb = np.linalg.norm(b, axis=-1)
    return np.clip((a * b).sum(-1) / np.maximum(na * nb, 1e-12), -1, 1)


def build_scalar_feats(X, noise_p, noise_s, noise_l=None):
    """속도·가속도·jerk·직진성·turn·노이즈 등 궤적 스칼라 피처."""
    delta_ = np.diff(X, axis=1)
    v_ = delta_ / DT
    a_ = np.diff(v_, axis=1) / DT
    jerk_ = np.diff(a_, axis=1) / DT
    sp_ = np.linalg.norm(v_, axis=-1)
    ac_ = np.linalg.norm(a_, axis=-1)
    jk_ = np.linalg.norm(jerk_, axis=-1)
    v_l = v_[:, -1, :]; a_l = a_[:, -1, :]
    a_r = a_[:, -3:, :].mean(axis=1)
    nd_vec = X[:, -1] - X[:, 0]; nd = np.linalg.norm(nd_vec, axis=-1)
    pl = np.linalg.norm(delta_, axis=-1).sum(axis=1)
    straight = np.where(pl > 1e-12, nd / np.maximum(pl, 1e-12), 0.0)
    turn = cos_safe(v_l, v_[:, :-1, :].mean(axis=1))
    if noise_l is None: noise_l = noise_s
    return pd.DataFrame({
        "mean_speed": sp_.mean(1), "max_speed": sp_.max(1),
        "speed_std": sp_.std(1), "mean_acc": ac_.mean(1),
        "max_acc": ac_.max(1), "max_jerk": jk_.max(1),
        "straightness": straight, "net_disp": nd,
        "turn_cos": turn, "|v_last|": np.linalg.norm(v_l, axis=-1),
        "|a_last|": np.linalg.norm(a_l, axis=-1),
        "|a_recent|": np.linalg.norm(a_r, axis=-1),
        "jerk_last": jk_[:, -1], "jerk_recent": jk_[:, -3:].mean(1),
        "noise_poly2": noise_p, "noise_savgol": noise_s, "noise_loo": noise_l,
        "hard_turn": (turn < 0.5).astype(np.float32),
        "high_speed": (np.linalg.norm(v_l, axis=-1) > 1.0).astype(np.float32),
        "high_acc": (ac_.max(axis=1) > 15).astype(np.float32),
        "log_max_acc": np.log1p(ac_.max(1)),
    })


def get_scalar_feats(X_train, X_test, mode_cfg, mode_name):
    """스칼라 피처 계산 + log1p 변환 (노이즈는 캐시)."""
    cache = CACHE_DIR / f"noise_{mode_name}.npz"
    if cache.exists():
        nc = np.load(cache)
        np_tr, ns_tr, nl_tr = nc["noise_p"], nc["noise_s"], nc["noise_l"]
        np_te, ns_te = nc["noise_p_test"], nc["noise_s_test"]
        print("[noise] cache 로드")
    else:
        print("[noise] 계산 중…")
        np_tr = noise_poly2(X_train); np_te = noise_poly2(X_test)
        ns_tr = noise_savgol(X_train); ns_te = noise_savgol(X_test)
        if mode_cfg["loo_sample"] is None:
            loo_idx = np.arange(X_train.shape[0])
        else:
            rng = np.random.RandomState(0)
            loo_idx = rng.choice(X_train.shape[0], size=mode_cfg["loo_sample"], replace=False)
        nl_tr = noise_loo_subset(X_train, loo_idx)
        np.savez(cache, noise_p=np_tr, noise_s=ns_tr, noise_l=nl_tr,
                  noise_p_test=np_te, noise_s_test=ns_te)
        print(f"[noise] cache 저장: {cache}")

    scal_tr = build_scalar_feats(X_train, np_tr, ns_tr, nl_tr)
    scal_te = build_scalar_feats(X_test,  np_te, ns_te)
    LOG_COLS = ["mean_speed","max_speed","speed_std","mean_acc","max_acc","max_jerk",
                "net_disp","|v_last|","|a_last|","|a_recent|","jerk_last","jerk_recent",
                "noise_poly2","noise_savgol","noise_loo"]
    for c in LOG_COLS:
        scal_tr[c] = np.log1p(scal_tr[c])
        scal_te[c] = np.log1p(scal_te[c])
    return scal_tr.values.astype(np.float32), scal_te.values.astype(np.float32)

In [5]:
def yaw_angle(v):
    """xy 진행방향(yaw) 각도."""
    return np.arctan2(v[:, 1], v[:, 0])


def rotate_xy(vec, theta):
    """xy 평면을 yaw로 회전 (z는 유지)."""
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([vec[:,0]*c + vec[:,1]*s,
                      -vec[:,0]*s + vec[:,1]*c,
                      vec[:,2]], axis=-1)


def inverse_rotate_xy(vec, theta):
    """rotate_xy의 역회전."""
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([vec[:,0]*c - vec[:,1]*s,
                      vec[:,0]*s + vec[:,1]*c,
                      vec[:,2]], axis=-1)


def build_tier3(X):
    """구간 평균 속도와 누적 이동거리 시퀀스 피처."""
    disp = np.diff(X, axis=1); v = disp / DT
    speed = np.linalg.norm(v, axis=-1)
    roll = np.stack([speed[:, i:i+3].mean(axis=1) for i in range(8)], axis=1) * DT
    cum = np.concatenate([np.zeros((X.shape[0], 1)),
                           np.cumsum(np.linalg.norm(disp, axis=-1), axis=1)], axis=1)
    return np.concatenate([roll, cum], axis=1).astype(np.float32)


def build_seq(X):
    """상대위치·속도·가속도를 쌓은 시퀀스 텐서 (N, T, 9)."""
    N = X.shape[0]
    rel = X - X[:, -1:, :]
    disp = np.diff(X, axis=1); v = disp / DT
    v_pad = np.concatenate([np.zeros((N,1,3)), v], axis=1)
    a = np.diff(v, axis=1) / DT
    a_pad = np.concatenate([np.zeros((N,2,3)), a], axis=1)
    return np.concatenate([rel, v_pad, a_pad], axis=-1).astype(np.float32)


def normalize_seq(arr, sc):
    """StandardScaler로 시퀀스를 정규화."""
    N, T, C = arr.shape
    return sc.transform(arr.reshape(-1, C)).astype(np.float32).reshape(N, T, C)

## §2. Pool A — Kalman 잔차 GRU (baseline)


In [6]:
def loss_euclid(pred, true):
    """유클리드 거리 손실."""
    return torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12).mean()


def loss_softhit(pred, true, beta=0.002):
    """1cm hit을 sigmoid로 근사한 soft-hit 손실."""
    d = torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12)
    return torch.sigmoid((d - 0.01) / beta).mean()


def loss_combo(pred, true):
    """유클리드 + soft-hit 결합 손실."""
    return loss_euclid(pred, true) + 0.3 * loss_softhit(pred, true)


class GRUModelMultiAux(nn.Module):
    """Kalman 잔차 GRU + scalar + 보조 head (Pool A 대표 모델)."""
    def __init__(self, n_channels=9, scal_dim=40, hidden=64, layers=1,
                 fc=128, p=0.2, aux_dims=(3, 3), main_scale_cm=2.0):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden, num_layers=layers, batch_first=True,
                           dropout=p if layers > 1 else 0)
        self.fc1 = nn.Linear(hidden + scal_dim, fc)
        self.fc2 = nn.Linear(fc, fc // 2)
        self.act = nn.GELU(); self.drop = nn.Dropout(p)
        self.head_main = nn.Linear(fc // 2, 3)
        self.aux_heads = nn.ModuleList([nn.Linear(fc // 2, d) for d in aux_dims])
        self.main_scale = main_scale_cm / 100.0

    def forward(self, seq, scal):
        out, _ = self.gru(seq)
        z = torch.cat([out[:, -1, :], scal], dim=1)
        z = self.act(self.fc1(z)); z = self.drop(z)
        z = self.act(self.fc2(z))
        out_main = torch.tanh(self.head_main(z)) * self.main_scale
        return out_main, [h(z) for h in self.aux_heads]

In [7]:
def run_kfold(target_main, target_F, target_W,
              seq_arr, scal_arr, seq_te, scal_te,
              kalman_train, theta_train, theta_test, y_train,
              config, mode_cfg, lambda_F=0.3, lambda_W=0.3, device="cpu"):
    """K-fold OOF 학습 루프 (시드 앙상블·early-stop·잔차→위치 복원)."""
    n_folds, n_seeds = mode_cfg["n_folds"], mode_cfg["n_seeds"]
    max_epochs, patience, batch = mode_cfg["max_epochs"], mode_cfg["patience"], mode_cfg["batch"]

    oof_rot = np.zeros((len(target_main), 3))
    fold_mask = np.zeros(len(target_main), dtype=bool)
    test_per_fold = []
    fold_rh = []
    kf = KFold(n_splits=max(n_folds, 2), shuffle=True, random_state=0)
    fold_iter = list(kf.split(np.arange(len(target_main))))
    if n_folds == 1:
        fold_iter = fold_iter[:1]
    t0 = time.time()

    for fi, (tr, va) in enumerate(fold_iter):
        sc_seq = StandardScaler().fit(seq_arr[tr].reshape(-1, seq_arr.shape[2]))
        sc_scal = StandardScaler().fit(scal_arr[tr])
        seq_tr_n = normalize_seq(seq_arr[tr], sc_seq)
        seq_va_n = normalize_seq(seq_arr[va], sc_seq)
        seq_te_n = normalize_seq(seq_te, sc_seq)
        scal_tr_n = sc_scal.transform(scal_arr[tr]).astype(np.float32)
        scal_va_n = sc_scal.transform(scal_arr[va]).astype(np.float32)
        scal_te_n = sc_scal.transform(scal_te).astype(np.float32)

        def T(a): return torch.from_numpy(a).to(device)
        seq_tr_t, scal_tr_t = T(seq_tr_n), T(scal_tr_n)
        tgt_tr_t = T(target_main[tr].astype(np.float32))
        F_tr_t   = T(target_F[tr].astype(np.float32))
        W_tr_t   = T(target_W[tr].astype(np.float32))
        seq_va_t, scal_va_t = T(seq_va_n), T(scal_va_n)
        seq_te_t, scal_te_t = T(seq_te_n), T(scal_te_n)

        seed_val, seed_test = [], []
        for s in range(n_seeds):
            torch.manual_seed(s); np.random.seed(s)
            model = GRUModelMultiAux(
                n_channels=seq_arr.shape[2], scal_dim=scal_arr.shape[1],
                hidden=config["hidden"], layers=config["layers"],
                fc=config["fc"], p=config["p"],
            ).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["wd"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

            best_rh, best_state, no_improve = -1.0, None, 0
            n_tr = seq_tr_t.shape[0]
            for ep in range(max_epochs):
                model.train()
                perm = torch.randperm(n_tr)
                for i in range(0, n_tr, batch):
                    idx = perm[i:i+batch]
                    opt.zero_grad()
                    out_main, outs_aux = model(seq_tr_t[idx], scal_tr_t[idx])
                    loss = loss_combo(out_main, tgt_tr_t[idx])
                    loss = loss + lambda_F * loss_euclid(outs_aux[0], F_tr_t[idx])
                    loss = loss + lambda_W * loss_euclid(outs_aux[1], W_tr_t[idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sched.step()

                model.eval()
                with torch.no_grad():
                    out_va_rot, _ = model(seq_va_t, scal_va_t)
                    out_va_rot = out_va_rot.cpu().numpy()
                out_va = inverse_rotate_xy(out_va_rot, theta_train[va])
                pred = kalman_train[va] + out_va
                rh = float((np.linalg.norm(pred - y_train[va], axis=-1) <= 0.01).mean())
                if rh > best_rh:
                    best_rh = rh
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= patience: break
                if ep == 0 or (ep + 1) % 5 == 0:
                    print(f"  fold{fi+1} seed{s} ep{ep+1:3d}: rhit={rh:.4f} (best {best_rh:.4f})  "
                          f"[{(time.time()-t0)/60:.1f}m]", flush=True)

            model.load_state_dict(best_state); model.eval()
            with torch.no_grad():
                pv_rot, _ = model(seq_va_t, scal_va_t)
                pt_rot, _ = model(seq_te_t, scal_te_t)
            seed_val.append(pv_rot.cpu().numpy())
            seed_test.append(pt_rot.cpu().numpy())
            del model; gc.collect()

        val_mean_rot = np.mean(seed_val, axis=0)
        test_mean_rot = np.mean(seed_test, axis=0)
        oof_rot[va] = val_mean_rot
        fold_mask[va] = True
        test_per_fold.append(test_mean_rot)

        val_unrot = inverse_rotate_xy(val_mean_rot, theta_train[va])
        pred_pos = kalman_train[va] + val_unrot
        rh_fold = float((np.linalg.norm(pred_pos - y_train[va], axis=-1) <= 0.01).mean())
        fold_rh.append(rh_fold)
        print(f"  ★ fold {fi+1}/{len(fold_iter)}: R-Hit={rh_fold:.4f}  ({(time.time()-t0)/60:.1f}m)", flush=True)

    if fold_mask.sum() == 0:
        oof_rhit = float("nan"); oof_unrot_all = np.zeros_like(target_main)
    else:
        oof_unrot_all = np.zeros_like(target_main)
        oof_unrot_all[fold_mask] = inverse_rotate_xy(oof_rot[fold_mask], theta_train[fold_mask])
        pred = kalman_train[fold_mask] + oof_unrot_all[fold_mask]
        oof_rhit = float((np.linalg.norm(pred - y_train[fold_mask], axis=-1) <= 0.01).mean())

    test_unrot = np.mean([inverse_rotate_xy(rot, theta_test) for rot in test_per_fold], axis=0)
    print(f"  OOF R-Hit (covered {fold_mask.sum()}): {oof_rhit:.4f}  소요 {(time.time()-t0)/60:.1f}m")
    return oof_unrot_all, test_unrot, fold_rh, oof_rhit, fold_mask

## §3. Pool B — Neural ODE (RK4 적분)


In [8]:
def rotate_xy_seq(seq_xyz: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """rotate (N, T, 3) by theta (N,) around z. y_local = -x*sin + y*cos."""
    c = np.cos(theta)[:, None]; s = np.sin(theta)[:, None]
    x_new = seq_xyz[..., 0] * c + seq_xyz[..., 1] * s
    y_new = -seq_xyz[..., 0] * s + seq_xyz[..., 1] * c
    return np.stack([x_new, y_new, seq_xyz[..., 2]], axis=-1).astype(np.float32)


# ============================================================
# Mirror aug (y-flip) — v90/v104 패턴
# ============================================================
def mirror_seq(seq):
    """물리적 좌우반사를 로컬 좌표로 변환한 mirror 증강."""
    out = seq.copy()
    out[..., 1] *= -1; out[..., 4] *= -1; out[..., 7] *= -1
    return out

def mirror_target(t):
    """y축 미러 증강 (타깃)."""
    out = t.copy(); out[..., 1] *= -1; return out


In [9]:
class ResBlock(nn.Module):
    """LayerNorm + residual MLP 블록."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(dim, dim),
        )
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        return self.ln(x + self.net(x))


class AccelField(nn.Module):
    """가속도 필드 a(pos, vel, latent, speed) → R^3."""
    def __init__(self, latent_dim=64, hidden=64):
        super().__init__()
        # pos(3) + vel(3) + latent + speed_scalar(1) = 7 + latent
        self.net = nn.Sequential(
            nn.Linear(3 + 3 + latent_dim + 1, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            ResBlock(hidden),
            nn.Linear(hidden, 3),
        )

    def forward(self, pos, vel, latent, speed):
        if speed.dim() == 1: speed = speed.unsqueeze(-1)
        x = torch.cat([pos, vel, latent, speed], dim=-1)
        return self.net(x)


class NeuralODEModel(nn.Module):
    """Position-velocity 6D state, RK4 single-step 80ms integration.

    Encoder: flatten(seq local-rotated) + scalar feats → latent z₀
    Dynamics: dp/dt = v ; dv/dt = -damping ⊙ v + a_neural(p,v,z,speed)
    Integration: RK4 (4 evals, dt=80ms)
    Output: final local position + local_bias → rotate back + last_obs + global_bias
    """
    def __init__(self, seq_dim=99, scal_dim=40, latent_dim=64, hidden=64,
                  dt_pred=0.080, n_steps=1):
        super().__init__()
        self.dt_pred = dt_pred
        self.n_steps = n_steps
        in_dim = seq_dim + scal_dim
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, latent_dim),
            nn.LayerNorm(latent_dim), nn.GELU(),
            ResBlock(latent_dim),
            ResBlock(latent_dim),
        )
        self.accel_field = AccelField(latent_dim=latent_dim, hidden=hidden)
        self.learned_damping = nn.Parameter(torch.tensor([0.1, 0.1, 0.1]))
        self.local_bias = nn.Parameter(torch.zeros(3))
        self.global_bias = nn.Parameter(torch.zeros(3))
        self._last_accels = []

    def _ode_deriv(self, pos, vel, latent, speed):
        a = self.accel_field(pos, vel, latent, speed)
        dp = vel
        dv = -self.learned_damping * vel + a
        return dp, dv, a

    def _rk4_step(self, pos, vel, latent, speed, dt):
        dp1, dv1, a1 = self._ode_deriv(pos, vel, latent, speed)
        pos2 = pos + 0.5 * dt * dp1; vel2 = vel + 0.5 * dt * dv1
        dp2, dv2, a2 = self._ode_deriv(pos2, vel2, latent, speed)
        pos3 = pos + 0.5 * dt * dp2; vel3 = vel + 0.5 * dt * dv2
        dp3, dv3, a3 = self._ode_deriv(pos3, vel3, latent, speed)
        pos4 = pos + dt * dp3; vel4 = vel + dt * dv3
        dp4, dv4, a4 = self._ode_deriv(pos4, vel4, latent, speed)
        new_pos = pos + (dt / 6.0) * (dp1 + 2 * dp2 + 2 * dp3 + dp4)
        new_vel = vel + (dt / 6.0) * (dv1 + 2 * dv2 + 2 * dv3 + dv4)
        return new_pos, new_vel, [a1, a2, a3, a4]

    def forward(self, seq_flat, scal, init_vel, speed):
        """Returns predicted local-frame displacement from last obs.

        seq_flat: (B, seq_dim) flattened normalized seq (local-rotated)
        scal:     (B, scal_dim)
        init_vel: (B, 3) initial velocity in local frame
        speed:    (B,) magnitude of init_vel (separate input for stability)
        """
        latent = self.backbone(torch.cat([seq_flat, scal], dim=-1))
        pos = torch.zeros_like(init_vel)
        vel = init_vel
        dt = self.dt_pred / self.n_steps
        accels = []
        for _ in range(self.n_steps):
            pos, vel, ac = self._rk4_step(pos, vel, latent, speed, dt)
            accels.extend(ac)
        self._last_accels = accels
        return pos + self.local_bias

In [10]:
def loss_huber(pred, true, delta=0.001):
    """Huber 손실."""
    return F.huber_loss(pred, true, delta=delta)

def loss_softhit(pred, true, k=300.0, c=0.01):
    """1cm hit을 sigmoid로 근사한 soft-hit 손실."""
    d = torch.sqrt(((pred - true) ** 2).sum(-1) + 1e-12)
    return torch.sigmoid((d - c) * k).mean()


def loss_combined(pred, true, accels, w_huber=100.0, w_hit=1.0, w_reg=1e-4):
    """Huber + soft-hit + 가속도 정규화 결합 손실."""
    huber = loss_huber(pred, true)
    hit = loss_softhit(pred, true)
    reg = 0.0
    if accels:
        reg = sum(a.pow(2).sum(-1).mean() for a in accels) / len(accels)
    return w_hit * hit + w_huber * huber + w_reg * reg, huber.item(), hit.item()

## §4. Pool C — Frenet 3D 프레임 + control-head


In [11]:
def yaw_frame(X):
    """yaw 회전 프레임 행렬 (저속/직선 fallback용)."""
    v = (X[:, -1] - X[:, -2]) / DT
    theta = yaw_angle(v)
    c, s = np.cos(theta), np.sin(theta)
    R = np.zeros((X.shape[0], 3, 3))
    # x' = c*x + s*y ; y' = -s*x + c*y ; z'=z   (matches rotate_xy)
    R[:, 0, 0] = c;  R[:, 0, 1] = s
    R[:, 1, 0] = -s; R[:, 1, 1] = c
    R[:, 2, 2] = 1.0
    return R

def frenet_frame(X):
    """tangent=last vel, normal from last accel (Gram-Schmidt), binormal=t×n.
    degenerate(저속/직선) 샘플은 yaw frame으로 fallback."""
    N = X.shape[0]
    v = (X[:, -1] - X[:, -2]) / DT                       # (N,3)
    a = (X[:, -1] - 2 * X[:, -2] + X[:, -3]) / (DT * DT)  # (N,3)
    nv = np.linalg.norm(v, axis=1)
    t = v / np.clip(nv[:, None], 1e-9, None)
    # normal = a - (a·t)t
    a_perp = a - (np.sum(a * t, axis=1, keepdims=True)) * t
    na = np.linalg.norm(a_perp, axis=1)
    n = a_perp / np.clip(na[:, None], 1e-9, None)
    b = np.cross(t, n)
    R = np.stack([t, n, b], axis=1)  # rows = frame axes → R@g = local
    # fallback: degenerate where |v|<1e-4 or |a_perp|<1e-6 → use yaw frame
    Ry = yaw_frame(X)
    bad = (nv < 1e-4) | (na < 1e-6)
    R[bad] = Ry[bad]
    return R

def apply_R_seq(seq, R):
    """seq (N,T,9)=[rel,v,a] 각 3-block에 R 적용 (local = R@vec)."""
    out = np.empty_like(seq)
    for k in range(3):
        blk = seq[..., 3*k:3*k+3]                       # (N,T,3)
        out[..., 3*k:3*k+3] = np.einsum("nij,ntj->nti", R, blk)
    return out.astype(np.float32)

def apply_R_vec(vec, R):       # (N,3) global→local
    return np.einsum("nij,nj->ni", R, vec).astype(np.float32)

def inv_R_vec(vec, R):         # (N,3) local→global  (R^T)
    return np.einsum("nji,nj->ni", R, vec)

def mirror_seq(seq, axis=1):
    """물리적 좌우반사를 local 좌표로 변환한 mirror.
    yaw frame: world-y negate -> local axis=1 (+ vel/accel blocks).
    frenet frame: 좌우반사 M=diag(1,-1,1)이 Frenet-local에서 binormal(z)만 negate
                  -> axis=2. (유도: local=(t·g, n·g, b·g), 반사 후 b_new=-Mb -> z성분 부호반전)"""
    out = seq.copy()
    for blk in range(3):
        out[..., 3 * blk + axis] *= -1
    return out

def mirror_vec(v, axis=1):
    """지정 축 반사 벡터."""
    out = v.copy(); out[..., axis] *= -1; return out

In [12]:
class GRUODEModel(nn.Module):
    """GRU 인코더 + Neural ODE 적분 (Pool C: Frenet/yaw 프레임용)."""
    def __init__(self, seq_channels=9, scal_dim=40, latent_dim=64, hidden=64,
                 dt_pred=0.080, n_steps=1):
        super().__init__()
        self.dt_pred = dt_pred; self.n_steps = n_steps
        self.gru = nn.GRU(seq_channels, latent_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.scal_proj = nn.Sequential(nn.Linear(scal_dim, latent_dim), nn.LayerNorm(latent_dim), nn.GELU())
        self.fuse = nn.Sequential(nn.Linear(2*latent_dim, latent_dim), nn.LayerNorm(latent_dim),
                                  nn.GELU(), ResBlock(latent_dim))
        self.accel_field = AccelField(latent_dim=latent_dim, hidden=hidden)
        self.learned_damping = nn.Parameter(torch.tensor([0.1, 0.1, 0.1]))
        self.local_bias = nn.Parameter(torch.zeros(3))
        self._last_accels = []

    def _ode_deriv(self, pos, vel, latent, speed):
        a = self.accel_field(pos, vel, latent, speed)
        return vel, -self.learned_damping * vel + a, a

    def _rk4(self, pos, vel, latent, speed, dt):
        dp1, dv1, a1 = self._ode_deriv(pos, vel, latent, speed)
        dp2, dv2, a2 = self._ode_deriv(pos + .5*dt*dp1, vel + .5*dt*dv1, latent, speed)
        dp3, dv3, a3 = self._ode_deriv(pos + .5*dt*dp2, vel + .5*dt*dv2, latent, speed)
        dp4, dv4, a4 = self._ode_deriv(pos + dt*dp3, vel + dt*dv3, latent, speed)
        np_ = pos + (dt/6)*(dp1+2*dp2+2*dp3+dp4)
        nv_ = vel + (dt/6)*(dv1+2*dv2+2*dv3+dv4)
        return np_, nv_, [a1, a2, a3, a4]

    def forward(self, seq, scal, init_vel, speed):
        # seq: (B,T,C)
        h = self.gru(seq)[0][:, -1]                     # last hidden (B,latent)
        latent = self.fuse(torch.cat([h, self.scal_proj(scal)], dim=-1))
        pos = torch.zeros_like(init_vel); vel = init_vel
        dt = self.dt_pred / self.n_steps; accels = []
        for _ in range(self.n_steps):
            pos, vel, ac = self._rk4(pos, vel, latent, speed, dt); accels.extend(ac)
        self._last_accels = accels
        return pos + self.local_bias

In [13]:
class ControlHead(nn.Module):
    """seq+scal -> latent -> control(accel[, jerk]); analytic integrate to +80ms disp."""
    def __init__(self, seq_dim, scal_dim, latent_dim=64, order="accel", T=0.080):
        super().__init__()
        self.T = T; self.order = order
        self.backbone = nn.Sequential(
            nn.Linear(seq_dim + scal_dim, latent_dim), nn.LayerNorm(latent_dim), nn.GELU(),
            ResBlock(latent_dim), ResBlock(latent_dim))
        nc = 3 if order == "accel" else 6  # accel | accel+jerk
        self.head = nn.Linear(latent_dim, nc)
        self.local_bias = nn.Parameter(torch.zeros(3))
        self.v_scale = nn.Parameter(torch.ones(1))   # learned trust on init velocity

    def forward(self, seq_flat, scal, init_vel, speed):
        z = self.backbone(torch.cat([seq_flat, scal], dim=-1))
        c = self.head(z); T = self.T
        a = c[:, :3]
        disp = self.v_scale * init_vel * T + 0.5 * a * (T * T)
        if self.order == "jerk":
            j = c[:, 3:6]; disp = disp + (1.0 / 6.0) * j * (T ** 3)
        return disp + self.local_bias

## §5. 회전물리 멤버 (HyperPhysics)

> 대회 중 코드 공유 게시판에 공개된 회전물리 baseline 구조를 참고(규칙 8조 B항). 가중치 차용 없이 train만으로 학습, test·외부데이터 미사용.


In [14]:
class SlidingWindowDataset(Dataset):
    """다중 타깃·가변 윈도 슬라이딩 데이터셋 (회전물리 학습용)."""
    def __init__(self, X, y, min_win=3, mode="extended", device="cpu", targets_ext=None):
        Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y, dtype=torch.float32)
        windows = []
        for i in range(len(X)):
            targets = (targets_ext or [4,5,6,7,8,9,10,12]) if mode == "extended" else [12, 10]
            for target_idx in targets:
                end_idx = target_idx - 2
                max_w = end_idx + 2 if mode == "extended" else (12 if target_idx == 12 else 10)
                for w in range(min_win, max_w):
                    windows.append((i, w, target_idx))
        Xl, yl = [], []
        for i, w, target_idx in windows:
            Xo = Xt[i]; end_idx = target_idx - 2
            pts = Xo[end_idx - w + 1: end_idx + 1]
            target = yt[i] if target_idx == 12 else Xo[target_idx]
            if w < 11:
                v0 = pts[1] - pts[0]; n_pad = 11 - w
                js = torch.arange(n_pad, 0, -1, dtype=torch.float32)
                pad = pts[0:1] - js.unsqueeze(1) * v0.unsqueeze(0)
                Xp = torch.cat([pad, pts], dim=0)
            else:
                Xp = pts.clone()
            Xl.append(Xp); yl.append(target)
        self.X_all = torch.stack(Xl).to(device); self.y_all = torch.stack(yl).to(device)
        diffs = self.X_all[:, 1:] - self.X_all[:, :-1]
        n1 = diffs[:, 1:].norm(dim=2).clamp(min=1e-8); n2 = diffs[:, :-1].norm(dim=2).clamp(min=1e-8)
        cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2) / (n1 * n2)).clamp(-1, 1)
        theta_last = torch.acos(cos_t[:, -1])
        self.theta_weights = (1.0 + 4.0 * (theta_last / 1.0).clamp(0, 1)).cpu().numpy()
    def __len__(self): return len(self.X_all)
    def __getitem__(self, idx): return self.X_all[idx], self.y_all[idx]

def _ema_va_local(diffs_local, alpha, beta):
    """EMA로 평활한 로컬 속도/가속도."""
    B, T, _ = diffs_local.shape; one_m_a = 1.0 - alpha; one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3); v = diffs_local[:, 0]; vs[:, 0] = v
    for t in range(1, T):
        v = alpha * diffs_local[:, t] + one_m_a * v; vs[:, t] = v
    vl = vs[:, -1]; ad = vs[:, 1:] - vs[:, :-1]; a = ad[:, 0]
    for t in range(1, T - 1):
        a = beta * ad[:, t] + one_m_b * a
    return vl, a

def _soft_hit_loss(pred, target, thr=0.013012, k=408.348):
    """soft-hit 손실 (회전물리용, sigmoid 근사)."""
    return (1 - torch.sigmoid(-(torch.norm(pred - target, dim=1) - thr) * k)).mean()

def extract_features(X, mean_stats, std_stats, dir_net, heading_mode="3step"):
    """회전물리용 로컬 프레임 피처 추출 (속도·가속도·각속도 등)."""
    device = X.device; p_last = X[:, 10]; diffs = X[:, 1:] - X[:, :-1]
    n1 = diffs[:, 1:].norm(dim=2, keepdim=True) + 1e-8; n2 = diffs[:, :-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2, keepdim=True) / (n1 * n2)).clamp(-1, 1)
    theta_seq = torch.acos(cos_t).squeeze(2)
    theta = theta_seq[:, -1:]; theta_mean = theta_seq.mean(1, keepdim=True); theta_std = theta_seq.std(1, keepdim=True)
    theta_vel = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc = theta_seq[:, -1:] - 2*theta_seq[:, -2:-1] + theta_seq[:, -3:-2]
    theta_trend = theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdim=True)
    if dir_net is not None:
        speed_seq = diffs.norm(dim=2); state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            state = torch.cat([state, diffs[:, :, 2].abs()], dim=1)
        weights = F.softmax(dir_net(state), dim=1); v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        v_sm = (3*diffs[:, -1] + 2*diffs[:, -2] + diffs[:, -3]) / 6.0 if heading_mode == "3step" else diffs[:, -1]
    fwd = v_sm / (v_sm.norm(dim=1, keepdim=True) + 1e-8)
    up_w = torch.zeros_like(fwd); up_w[:, 2] = 1.0
    up_w[fwd[:, 2].abs() > 0.99] = torch.tensor([0., 1., 0.], device=device)
    right = torch.cross(fwd, up_w, dim=1); right = right / (right.norm(dim=1, keepdim=True) + 1e-8)
    up = torch.cross(right, fwd, dim=1); up = up / (up.norm(dim=1, keepdim=True) + 1e-8)
    R = torch.stack([fwd, right, up], dim=2)
    v_last = diffs[:, -1]; v_prev1 = diffs[:, -2]; speed = v_last.norm(dim=1, keepdim=True)
    a_last = v_last - v_prev1; acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1); a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)
    X_local = torch.matmul(X - p_last.unsqueeze(1), R); p_std_local = X_local.std(1); v_local_abs = v_local.abs()
    jerk_g = diffs[:, -1] - 2*diffs[:, -2] + diffs[:, -3]; jerk_l = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1)
    jerk_mag = jerk_g.norm(dim=1, keepdim=True)
    features = torch.cat([v_local, a_local, speed, acc_mag, theta, theta_mean, theta_std, theta_trend,
                          theta_vel, theta_acc, p_std_local, v_local_abs, jerk_l, jerk_mag], dim=1)
    if mean_stats is None or std_stats is None:
        mean_stats = features.mean(0, keepdim=True); std_stats = features.std(0, keepdim=True) + 1e-8
    return (features - mean_stats) / std_stats, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats

In [15]:
class ResBlock(nn.Module):
    """LayerNorm + residual MLP 블록."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15), nn.Linear(dim, dim))
        self.ln = nn.LayerNorm(dim)
    def forward(self, x): return self.ln(x + self.net(x))

class PriorBiasedLinear(nn.Module):
    """prior bias를 더하는 선형층 (weight 0 초기화)."""
    def __init__(self, in_f, out_f, prior_bias):
        super().__init__(); self.linear = nn.Linear(in_f, out_f)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad(): nn.init.zeros_(self.linear.weight); nn.init.zeros_(self.linear.bias)
    def forward(self, x): return self.linear(x) + self.prior_bias

def rodrigues_rotate(v, w):
    """Rodrigues 공식으로 벡터를 각속도만큼 회전."""
    theta = w.norm(dim=1, keepdim=True); k = w / (theta + 1e-8)
    cos_t = torch.cos(theta); sin_t = torch.sin(theta); dot = (v * k).sum(dim=1, keepdim=True); cross = torch.cross(k, v, dim=1)
    return v * cos_t + cross * sin_t + k * dot * (1.0 - cos_t)

class HyperPhysics_xy2(nn.Module):
    """회전 기반 turn-rate 물리모델 (회전물리 멤버, HyperPhysics)."""
    def __init__(self, input_dim=24, **kw):
        super().__init__()
        self.sh_thr=kw.pop('sh_thr',0.013012); self.sh_k=kw.pop('sh_k',408.348044); self.mse_w=kw.pop('mse_w',129.172037)
        self.local_w=kw.pop('local_w',0.050941); self.theta_thr=kw.pop('theta_thr',1.087618); self.speed_thr=kw.pop('speed_thr',0.034583)
        self.lr=0.005400; self.wd=0.005659
        self.register_buffer("mean_stats", torch.zeros(1, input_dim)); self.register_buffer("std_stats", torch.ones(1, input_dim))
        self.use_dirnet = True   # False면 고정 3-step heading (다른 프레임 → decorrelated 2nd 물리)
        prior_dir = torch.tensor([-10.,-10.,-10.,-10.,-10.,-10.,-10.,0.,0.693,1.098])
        self.dir_net = nn.Sequential(nn.Linear(29,24), nn.LayerNorm(24), nn.GELU(), PriorBiasedLinear(24,10,prior_dir))
        self.temporal_net = nn.Sequential(nn.Linear(9,32), nn.LayerNorm(32), nn.GELU(), PriorBiasedLinear(32,6,torch.zeros(6)))
        prior_dyn = torch.tensor([0.,0.,0.,0.,0.,0.]+[-4.]*24)
        self.dynamics_net = nn.Sequential(nn.Linear(input_dim,96), nn.LayerNorm(96), nn.GELU(), ResBlock(96), PriorBiasedLinear(96,30,prior_dyn))
        self.omega_w = nn.Parameter(torch.tensor([0.0,-0.5,-1.0]))
        self.omega_net = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim,48), nn.GELU(), nn.Linear(48,3))
        with torch.no_grad(): nn.init.normal_(self.omega_net[-1].weight, std=0.01); nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(nn.Linear(input_dim,32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32,3))
    def get_features(self, X, mean_stats=None, std_stats=None):
        return extract_features(X, mean_stats, std_stats, self.dir_net if self.use_dirnet else None, "3step")
    @staticmethod
    def _rotation_vector(d_prev, d_curr):
        n_prev = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8); n_curr = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hat_prev = d_prev/n_prev; d_hat_curr = d_curr/n_curr; cross = torch.linalg.cross(d_hat_prev, d_hat_curr, dim=1)
        sin_t = cross.norm(dim=1, keepdim=True).clamp(min=1e-8); cos_t = (d_hat_prev*d_hat_curr).sum(1, keepdim=True).clamp(-0.9999, 0.9999)
        theta = torch.atan2(sin_t, cos_t); speed_gate = torch.sigmoid((n_prev+n_curr)*500-5)
        return cross/sin_t*theta*speed_gate
    def forward(self, features, diffs, p_last, theta, speed, R):
        B = diffs.shape[0]
        ema_raw = self.temporal_net(features[:, 8:17]); alpha = torch.sigmoid(ema_raw[:, 0:3])*0.8+0.1; beta = torch.sigmoid(ema_raw[:, 3:6])*0.199+0.8
        dyn_raw = self.dynamics_net(features); w_v = 2.0+dyn_raw[:, 0:3]; w_a = 1.0+dyn_raw[:, 3:6]
        v_local_abs = features[:, 17:20]; v_local_abs2 = v_local_abs*v_local_abs; theta2 = theta*theta
        exp_v = (F.softplus(dyn_raw[:,6:9])*v_local_abs + F.softplus(dyn_raw[:,9:12])*v_local_abs2 + F.softplus(dyn_raw[:,12:15])*theta + F.softplus(dyn_raw[:,15:18])*theta2)
        exp_a = (F.softplus(dyn_raw[:,18:21])*v_local_abs + F.softplus(dyn_raw[:,21:24])*v_local_abs2 + F.softplus(dyn_raw[:,24:27])*theta + F.softplus(dyn_raw[:,27:30])*theta2)
        diffs_local = torch.matmul(diffs, R); vl, al = _ema_va_local(diffs_local, alpha, beta); diff_speed = diffs_local.norm(dim=2)
        def rv_masked(ka, kb):
            rv = self._rotation_vector(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv*valid.unsqueeze(1), valid
        ov1, vm1 = rv_masked(-2,-1); ov2, vm2 = rv_masked(-3,-2); ov3, vm3 = rv_masked(-4,-3)
        w_logits = self.omega_w.view(1,3).expand(B,-1); masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_attn = F.softmax(w_logits.masked_fill(masks == 0, -1e9), dim=1)
        omega_hist = w_attn[:,0].unsqueeze(1)*ov1 + w_attn[:,1].unsqueeze(1)*ov2 + w_attn[:,2].unsqueeze(1)*ov3
        current_speed = speed.view(B,1) if speed is not None else diff_speed[:,-1].unsqueeze(1)
        omega_delta = self.omega_net(features) * torch.sigmoid(current_speed*500-5)
        theta_scalar = theta.view(B,1)
        rotation_gate = torch.sigmoid((theta_scalar-self.theta_thr)*10) * torch.sigmoid((current_speed-self.speed_thr)*200)
        omega = (omega_hist + omega_delta) * rotation_gate
        v_rotated = rodrigues_rotate(vl, omega)
        pred_local = (w_v*torch.exp(-exp_v))*v_rotated + (w_a*torch.exp(-exp_a))*al
        log_var = self.diffusion_net(features).clamp(min=-5.0, max=5.0)
        pred_global = p_last + torch.einsum('nij,nj->ni', R, pred_local)
        return pred_global, pred_local, log_var
    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, **kw):
        loss = _soft_hit_loss(pp, yr, self.sh_thr, self.sh_k) + self.mse_w*F.mse_loss(pp, yr)
        if pred_local is not None and yr_local is not None and log_var is not None:
            nll = 0.5*(torch.exp(-log_var)*(pred_local-yr_local)**2 + log_var); loss = loss + self.local_w*nll.mean()
        return loss

In [16]:
@torch.no_grad()
def predict_full(model, X, device):
    """배치 추론으로 전체 입력의 예측을 반환."""
    Xt = torch.tensor(X, dtype=torch.float32, device=device); preds = []
    for i in range(0, len(Xt), 512):
        b = Xt[i:i+512]
        ft, df, pl, th, _, _, _, R, sp, _, _ = model.get_features(b, model.mean_stats, model.std_stats)
        pp, _, _ = model(ft, df, pl, th, sp, R); preds.append(pp.cpu().numpy())
    return np.concatenate(preds, 0)

def hit(p, y): return float((np.linalg.norm(p - y, axis=-1) <= 0.01).mean())

def train_fold(Xtr, ytr, epochs, min_win, targets_ext, device, use_dirnet=True):
    """R-Hit@1cm (예측-정답 거리 ≤ 1cm 비율)."""
    ds = SlidingWindowDataset(Xtr, ytr, min_win=min_win, mode="extended", device=device, targets_ext=targets_ext)
    sampler = WeightedRandomSampler(ds.theta_weights, len(ds), replacement=True)
    loader = DataLoader(ds, batch_size=256, sampler=sampler)
    model = HyperPhysics_xy2().to(device); model.use_dirnet = use_dirnet
    with torch.no_grad():
        _, _, _, _, _, _, _, _, _, mn, st = model.get_features(torch.tensor(Xtr, dtype=torch.float32, device=device))
        model.mean_stats.copy_(mn); model.std_stats.copy_(st)
    opt = torch.optim.AdamW(model.parameters(), lr=model.lr, weight_decay=model.wd)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    for ep in range(1, epochs+1):
        model.train(); tl = 0.0
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            ft, df, pl, th, _, _, _, R, sp, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)
            pp, prl, lv = model(ft, df, pl, th, sp, R)
            yr_local = torch.matmul((yb - pl).unsqueeze(1), R).squeeze(1)
            loss = model.compute_loss(pp, yb, prl, yr_local, lv)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl += loss.item()*len(Xb)
        sch.step()
        if ep <= 3 or ep % 4 == 0: print(f"    ep{ep}/{epochs} loss={tl/len(ds):.4f}", flush=True)
    return model

## §6. DE 블렌드(base) + 최종 α 주입

```text
base       = §2-§5 멤버 DE conservative 블렌드 (OOF 0.6831) — 예측을 §7에 압축 내장
phys_ens3  = 회전물리 3-멤버 (HyperPhysics) 평균
final      = 0.60·base + 0.40·phys_ens3
```

base는 멤버 ~40개의 앙상블 결과라 단일 실행 재학습이 비현실적(15-20시간)이어서, 그 test 예측을 §7에 내장한다. 회전물리 멤버는 §5 코드로 data/에서 직접 재학습도 가능(§7 `RETRAIN_PHYS=True`).


## §7. 최종 제출 복원 (단일 .ipynb · 외부 파일 불필요)

내장된 base·회전물리 예측을 0.60 : 0.40으로 블렌드해 `submission_v157_ens3a0.4.csv`(Private 0.7031)를 생성한다. 외부 데이터·산출물 없이 바로 실행된다.

- `RETRAIN_PHYS=True`로 두면 §5 코드로 회전물리 3멤버를 data/에서 직접 5-fold 재학습해 사용(점수 ≈ 동일).


In [17]:
# ===== §7. 최종 제출(Private 0.7031, 2위) 복원 — 단일 .ipynb, 외부 파일 불필요 =====
# base(40멤버 블렌드)·회전물리 3-앙상블 test 예측을 노트북에 압축 내장(zlib+base64).
_BASE_B64 = "..."
_PHYS_B64 = "..."

import zlib, base64
ALPHA_FINAL  = 0.40
RETRAIN_PHYS = False   # False: 내장 예측으로 즉시 복원 (외부 파일 0개, <1초)
                       # True : §5 코드로 회전물리 3멤버를 data/에서 직접 5-fold 재학습 (점수 ≈ 동일)


def _decode(b64, shape=(10000, 3)):
    """노트북 내장 예측(zlib + base64, float32) 복원."""
    arr = np.frombuffer(zlib.decompress(base64.b64decode(b64)), dtype=np.float32)
    return arr.reshape(shape).astype(np.float64)


def phys_member(tag, seed, heading, epochs=14, min_win=5, aug="reduced"):
    """회전물리(HyperPhysics) 멤버 5-fold OOF 학습 (§5 train_fold/predict_full). RETRAIN_PHYS 전용."""
    seed_everything(seed); dev = torch.device("cpu"); torch.set_num_threads(os.cpu_count() or 4)
    targets_ext = [4, 5, 6, 7, 8, 9, 10, 12] if aug == "full" else [6, 7, 8, 9, 10, 12]
    N = len(X_train); kf = KFold(n_splits=5, shuffle=True, random_state=0); test_acc = []
    for tr, va in kf.split(np.arange(N)):
        m = train_fold(X_train[tr], y_train[tr], epochs, min_win, targets_ext, dev,
                       use_dirnet=(heading == "dirnet")).eval()
        test_acc.append(predict_full(m, X_test, dev))
    return np.mean(test_acc, axis=0).astype(np.float64)


# base = §2~§5 멤버 ~40개의 DE conservative 블렌드(OOF 0.6831) 결과.
# 멤버 전체 재학습은 비현실적(15~20h)이라, 그 test 예측을 노트북에 압축 내장(위 _BASE_B64).
base = _decode(_BASE_B64)

if RETRAIN_PHYS:
    X_train, X_test, y_train, sub = load_data()
    SPECS = [("xy2", 42, "dirnet"), ("xy2s1", 1, "dirnet"), ("xy2h3", 42, "3step")]
    phys_ens3 = np.mean([phys_member(t, s, h) for t, s, h in SPECS], axis=0)
    ids = sub["id"].tolist()
else:
    phys_ens3 = _decode(_PHYS_B64)                      # 회전물리 3-앙상블 내장 예측
    ids = [f"TEST_{i:05d}" for i in range(1, 10001)]    # 제출 id (sample_submission 순서)

# 최종 α 주입 -> 제출 파일 생성
final = (1.0 - ALPHA_FINAL) * base + ALPHA_FINAL * phys_ens3
out = DATA_DIR / "submission_v157_ens3a0.4.csv"; out.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": ids, "x": final[:, 0], "y": final[:, 1], "z": final[:, 2]}).to_csv(out, index=False)
print(f"[done] 최종 제출 생성: {out.name}  (rows={len(final)})  -> Private 0.7031 (2위)")

[done] 최종 제출 생성: submission_v157_ens3a0.4.csv  (rows=10000)  -> Private 0.7031 (2위)
